In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
import tensorflow as tf
from tensorflow import keras

fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

train_images = train_images / 255.0
test_images = test_images / 255.0

model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10)
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(train_images, train_labels, epochs=5, verbose=1)

probability_model = keras.Sequential([model, keras.layers.Softmax()])
predictions = probability_model.predict(test_images, verbose=0)

idxs = [0, 1, 2, 3, 4, 5]
images = test_images[idxs]
true_labels = test_labels[idxs]
pred_probs = predictions[idxs]
pred_labels = np.argmax(pred_probs, axis=1)

frames_per_image = 18
n_images = len(idxs)
total_frames = n_images * frames_per_image

fig = plt.figure(figsize=(10, 5.3), facecolor='white')
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.25])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

plt.subplots_adjust(top=0.82, bottom=0.16, wspace=0.35)

def draw_frame(frame):
    ax1.clear()
    ax2.clear()
    ax3.clear()

    img_idx = frame // frames_per_image
    local_frame = frame % frames_per_image

    img = images[img_idx]
    true_label = true_labels[img_idx]
    probs = pred_probs[img_idx]
    pred_label = pred_labels[img_idx]

    fig.suptitle("Da imagem à classificação com Fashion-MNIST",
                 fontsize=16, fontweight='bold', y=0.95)

    if local_frame < 6:
        ax1.imshow(img, cmap='viridis')
        ax1.set_title("Mapa de calor", fontsize=11)
        ax2.axis('off')
        ax3.axis('off')

    elif local_frame < 12:
        ax1.imshow(img, cmap='viridis')
        ax1.set_title("Mapa de calor", fontsize=11)

        ax2.imshow(img, cmap='gray')
        ax2.set_title("Imagem real", fontsize=11)

        ax3.axis('off')

    else:
        ax1.imshow(img, cmap='viridis')
        ax1.set_title("Mapa de calor", fontsize=11)

        ax2.imshow(img, cmap='gray')
        ax2.set_title("Imagem real", fontsize=11)

        ax3.barh(range(10), probs)
        ax3.set_yticks(range(10))
        ax3.set_yticklabels(class_names, fontsize=8)
        ax3.set_xlim([0, 1])

        ax3.set_xlabel("Probabilidade", fontsize=10, labelpad=8)
        ax3.set_title(
            f"Real: {class_names[true_label]} | Predito: {class_names[pred_label]} ({probs[pred_label]*100:.1f}%)",
            fontsize=9.5,
            fontweight='bold',
            pad=8
        )

        ax3.get_yticklabels()[pred_label].set_fontweight('bold')

    for ax in [ax1, ax2]:
        ax.set_xticks([])
        ax.set_yticks([])

anim = FuncAnimation(fig, draw_frame, frames=total_frames, interval=250)

anim.save("fashion_mnist_classificacao.gif", writer=PillowWriter(fps=4))
anim.save("fashion_mnist_classificacao.mp4", writer=FFMpegWriter(fps=4))

print("Arquivos salvos: fashion_mnist_classificacao.gif e fashion_mnist_classificacao.mp4")